In [1]:
!pip install peft trl -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.8/374.8 kB 9.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.4/293.4 kB 20.8 MB/s eta 0:00:00


In [ ]:
#!pip install bitsandbytes peft trl -U
#export HF_ENDPOINT="https://hf-mirror.com"
import os

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HUB_ENABLE_HF_MIRROR"] = "true"

# 文语义理解

* microsoft/Phi-3 mini 3.8B small 7B medium 14B "microsoft/Phi-3-small-8k-instruct"

## 加载数据

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

#model_id = "/kaggle/input/phi-3/pytorch/phi-3.5-mini-instruct/2"
model_id = "microsoft/phi-4"

tokenizer=AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

tokenizer_config.json:   0%|          | 0.00/17.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.61M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/917k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.25M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

In [7]:
from datasets import load_dataset

# 中科院中国传统文化数据集
#raw_data=load_dataset("m-a-p/COIG-CQIA", 'chinese_traditional' , split="train[:500]")
#raw_data=raw_data.train_test_split(test_size=0.1)
raw_data=load_dataset("m-a-p/COIG-CQIA", 'chinese_traditional' )
#test_data=load_dataset("m-a-p/COIG-CQIA", 'chinese_traditional' , "test")
print(raw_data["train"][0])
raw_data

{'instruction': '暗度陈仓\n成语释义：', 'input': '', 'output': '指正面迷惑敌人，而从侧翼进行突然袭击。亦比喻暗中进行活动。陈仓，古县名，在今陕西省宝鸡市东，为通向汉中的交通孔道。\n成语出处：《史记·高祖本纪》……项王使卒三万人从，楚与诸侯之慕从者数万人，从杜南入蚀中。去辄烧绝栈道，以备诸侯盗兵袭之，亦示项羽无东意……八月，汉王用韩信之计，从故道还，袭雍王章邯—迎击汉陈仓，雍兵败，…', 'task_type': {'major': ['文本生成'], 'minor': ['成语释义']}, 'domain': ['中国传统文化', '成语'], 'metadata': '暂无元数据信息', 'answer_from': 'human', 'human_verified': True, 'copyright': '暂无版权及作者信息'}


DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'task_type', 'domain', 'metadata', 'answer_from', 'human_verified', 'copyright'],
        num_rows: 1111
    })
})

In [8]:
def preprocess_func(example, tokenize_enable=False):
    output_texts = []
    for i in range(len(example['instruction'])):
        messages = [
            {"role": "system", "content": "您是中文智能问答助手！"},
            {"role": "user", "content": "{}".format(example['instruction'][i])},
            {"role": "assistant", "content": "{}".format(example['output'][i])}
        ]
        output_texts.append(
            tokenizer.apply_chat_template(messages, tokenize=tokenize_enable, add_generation_prompt=False)
        )
        
    return output_texts

# Print the first training example
print(preprocess_func(raw_data["train"][:1], tokenize_enable=False)[0])

<|im_start|>system<|im_sep|>您是中文智能问答助手！<|im_end|><|im_start|>user<|im_sep|>暗度陈仓
成语释义：<|im_end|><|im_start|>assistant<|im_sep|>指正面迷惑敌人，而从侧翼进行突然袭击。亦比喻暗中进行活动。陈仓，古县名，在今陕西省宝鸡市东，为通向汉中的交通孔道。
成语出处：《史记·高祖本纪》……项王使卒三万人从，楚与诸侯之慕从者数万人，从杜南入蚀中。去辄烧绝栈道，以备诸侯盗兵袭之，亦示项羽无东意……八月，汉王用韩信之计，从故道还，袭雍王章邯—迎击汉陈仓，雍兵败，…<|im_end|>


## 转换训练标记

In [9]:
import matplotlib.pyplot as plt

dataset_tokenized = preprocess_func(raw_data["train"], tokenize_enable=True) + preprocess_func(raw_data["test"], tokenize_enable=True)
data=[len(tok) for tok in dataset_tokenized]
print(f"Longest sample: {max(data)} tokens")

plt.hist(data, bins=10)
plt.show()

KeyError: 'test'

## 加载量化模型

In [10]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

model=AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

model.config.eos_token_id=tokenizer.eos_token_id
model.gradient_checkpointing_enable() # reducing memory usage
print(model.model.embed_tokens)

config.json:   0%|          | 0.00/820 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.4k [00:00<?, ?B/s]

model-00001-of-00006.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

model-00002-of-00006.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00003-of-00006.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

...

model-00005-of-00006.safetensors:   0%|          | 0.00/4.77G [00:00<?, ?B/s]

model-00006-of-00006.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

Embedding(100352, 5120, padding_idx=100257)


In [25]:
import transformers

pipeline = transformers.pipeline(
    "text-generation",
    model,
    model_kwargs={"torch_dtype": "auto"},
    device_map="auto",tokenizer=tokenizer
)

messages = [
    {"role": "system", "content": "您扮演女医生."},
    {"role": "user", "content": "备孕需要做说明?"},
]

outputs = pipeline(messages, max_new_tokens=512)
print(outputs[0]["generated_text"][-1])

{'role': 'assistant', 'content': '备孕是一个需要一些准备和注意的过程，以下是一些常见的建议和注意事项：\n\n1. **营养均衡**：\n   - 确保摄入足够的维生素和矿物质，特别是叶酸、铁、维生素D和维生素B群。\n   - 保持均衡的饮食，包括蔬菜、水果、全谷物、蛋白质和健康脂肪。\n\n2. **体重管理**：\n   - 保持健康的体重。过重或过轻都可能影响受孕能力。\n\n3. **戒烟、戒酒**：\n   - 吸烟和饮酒都可能影响受孕能力和胎儿发育，建议戒烟和限制酒精摄入。\n\n4. **定期锻炼**：\n   - 适量的运动有助于保持健康的体重和心理状态，但避免过度运动。\n\n5. **减少压力**：\n   - 压力可能影响生育能力，尝试通过冥想、瑜伽或其他放松技巧来管理压力。\n\n6. **避免有害物质**：\n   - 避免接触有毒化学物质、某些药物和辐射。\n\n7. **医疗检查**：\n   - 建议进行全面的健康检查，包括性传播疾病检测、血液检查和必要的身体检查。\n   - 如果有慢性病（如糖尿病、高血压等），确保病情得到良好控制。\n\n8. **'}


In [26]:
from IPython.display import Markdown
response = outputs[0]["generated_text"][-1]
Markdown(response['content'])

备孕是一个需要一些准备和注意的过程，以下是一些常见的建议和注意事项：

1. **营养均衡**：
   - 确保摄入足够的维生素和矿物质，特别是叶酸、铁、维生素D和维生素B群。
   - 保持均衡的饮食，包括蔬菜、水果、全谷物、蛋白质和健康脂肪。

2. **体重管理**：
   - 保持健康的体重。过重或过轻都可能影响受孕能力。

3. **戒烟、戒酒**：
   - 吸烟和饮酒都可能影响受孕能力和胎儿发育，建议戒烟和限制酒精摄入。

4. **定期锻炼**：
   - 适量的运动有助于保持健康的体重和心理状态，但避免过度运动。

5. **减少压力**：
   - 压力可能影响生育能力，尝试通过冥想、瑜伽或其他放松技巧来管理压力。

6. **避免有害物质**：
   - 避免接触有毒化学物质、某些药物和辐射。

7. **医疗检查**：
   - 建议进行全面的健康检查，包括性传播疾病检测、血液检查和必要的身体检查。
   - 如果有慢性病（如糖尿病、高血压等），确保病情得到良好控制。

8. **

## 冻结低秩优化

In [11]:
def print_trainable_parameters(model):
    trainable_params=0
    all_params=0
    for _, param in model.named_parameters():
        all_params+=param.numel()
        if param.requires_grad:
            trainable_params+=param.numel()
    print(f"trainable params: {trainable_params} || all params: {all_params} || trainable%: {100 * trainable_params/all_params:.2f}")

print_trainable_parameters(model)

trainable params: 14659507200 || all params: 14659507200 || trainable%: 100.00


In [12]:
from peft import prepare_model_for_kbit_training

prepared_model=prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=True
)

print_trainable_parameters(prepared_model)
print(prepared_model)

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.91 GiB. GPU 0 has a total capacity of 14.74 GiB of which 1.58 GiB is free. Process 2694 has 13.15 GiB memory in use. Of the allocated memory 13.06 GiB is allocated by PyTorch, and 1.62 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

lora_config=LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=['qkv_proj', 'o_proj'],
    lora_dropout=0.1,
    bias="none",
    modules_to_save=["lm_head","embed_tokens"], # we added new tokens to tokenizer, this is necesarry
    task_type=TaskType.CAUSAL_LM
)

lora_model=get_peft_model(prepared_model, lora_config)
lora_model.config.use_cache=False
print_trainable_parameters(lora_model)
print(lora_model)

## 训练模型

In [ ]:
import transformers
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=raw_data['train'],
    #max_seq_length=512,
    tokenizer=tokenizer,
    args=transformers.TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=2,
        max_steps=50,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=5,
        report_to='none',
        output_dir='logs',
        optim="paged_adamw_8bit"
    ),
    peft_config=lora_config,
    formatting_func=formatting_prompts_func,
)

trainer.train()

In [ ]:
trainer.model.save_pretrained('lora_adapter')

In [ ]:
from peft import LoraConfig, PeftModel
base_model_id = "/kaggle/input/phi-3/pytorch/phi-3.5-mini-instruct/2"
adapter_model_id = "/kaggle/working/lora_adapter"

base_model = AutoModelForCausalLM.from_pretrained(base_model_id, device_map='auto', torch_dtype=torch.float16)
merge_model = PeftModel.from_pretrained(base_model, adapter_model_id, device_map='auto', torch_dtype=torch.float16)

# Merge the adapters into the base model so you can use the model like a normal transformers model
model = merge_model.merge_and_unload()
model.save_pretrained('/tmp/final_model')
tokenizer.save_pretrained('/tmp/final_model')

# 推理

In [ ]:
import gc

del tokenizer, lora_model, prepared_model, model, trainer, base_model, merge_model

gc.collect()
torch.cuda.empty_cache()

In [ ]:
messages = [
    {
        "role": "user",
        "content": "飞龙在天"
    }
]

## 基模型

In [ ]:
from transformers import pipeline, AutoTokenizer

model_old_id = "/kaggle/input/phi-3/pytorch/phi-3.5-mini-instruct/2"

tokenizer = AutoTokenizer.from_pretrained(model_old_id)
pipe = pipeline(
    "text-generation",
    model=model_old_id,
    tokenizer=tokenizer,
    model_kwargs={"torch_dtype": torch.float16},
    device_map='auto',
    max_new_tokens=512
)

In [ ]:
from IPython.display import Markdown 

prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
outputs = pipe(
    prompt,
    do_sample=True,
    temperature=0.1,
    top_k=20,
    top_p=0.3,
    add_special_tokens=True
)

display(Markdown(outputs[0]["generated_text"][len(prompt):].replace('#', '')))

## 新模型

In [ ]:
from transformers import pipeline, AutoTokenizer

model_new_id = "/tmp/final_model"
tokenizer = AutoTokenizer.from_pretrained(model_new_id)
pipe_finetuned = pipeline(
    "text-generation",
    model=model_new_id,
    tokenizer=tokenizer,
    model_kwargs={"torch_dtype": torch.float16},
    device_map='auto',
    max_new_tokens=512
)

In [ ]:
from IPython.display import Markdown 

prompt = pipe_finetuned.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
outputs = pipe_finetuned(
    prompt,
    do_sample=True,
    temperature=0.1,
    top_k=20,
    top_p=0.3,
    add_special_tokens=True
)

display(Markdown(outputs[0]["generated_text"][len(prompt):].replace('#', '')))

# 图语义理解

In [ ]:
from PIL import Image 
import requests
from transformers import AutoModelForCausalLM 
from transformers import AutoProcessor 

model_id = "/kaggle/input/phi-3/pytorch/phi-3.5-vision-instruct/1" 
#model_id = "microsoft/Phi-3.5-vision-instruct" 
# Note: set _attn_implementation='eager' if you don't have flash_attn installed

bnb_config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    llm_int8_enable_fp32_cpu_offload=True,
)

model = AutoModelForCausalLM.from_pretrained(
  model_id, quantization_config=bnb_config,
  device_map="auto", 
  trust_remote_code=True,
  _attn_implementation='eager'
)

# for best performance, use num_crops=4 for multi-frame, num_crops=16 for single-frame.
processor = AutoProcessor.from_pretrained(model_id, 
  trust_remote_code=True, 
  num_crops=4
) 

model

In [ ]:
images = []
placeholder = ""

# Note: if OOM, you might consider reduce number of frames in this example.
for i in range(1,3):
    url = f"https://image.slidesharecdn.com/azureintroduction-191206101932/75/Introduction-to-Microsoft-Azure-Cloud-{i}-2048.jpg" 
    images.append(Image.open(requests.get(url, stream=True).raw))
    placeholder += f"<|image_{i}|>\n"

In [ ]:
messages = [
    {"role": "user", "content": placeholder+"Summarize the deck of slides."},
]

prompt = processor.tokenizer.apply_chat_template(
  messages, 
  tokenize=False, 
  add_generation_prompt=True
)

inputs = processor(prompt, images, return_tensors="pt").to("cuda:0") 

generation_args = { 
    "max_new_tokens": 1000, 
    "temperature": 0.0, 
    "do_sample": False, 
} 

generate_ids = model.generate(**inputs, 
  eos_token_id=processor.tokenizer.eos_token_id, 
  **generation_args
)

# remove input tokens 
generate_ids = generate_ids[:, inputs['input_ids'].shape[1]:]
response = processor.batch_decode(generate_ids, 
  skip_special_tokens=True, 
  clean_up_tokenization_spaces=False)[0] 

print(response)